# Занятие 4. Строки

**План занятия**

1. Индексы и срезы
2. Неизменяемость и цена склейки
3. Методы и проверки
4. `split` и `join`
5. Разбор строки на поля
6. Домашние задачи

**Теория:** `theory/04_Строки.md`
Т. Гэддис, гл. 8 (с. 436 / PDF 461)

In [ ]:
# Эта ячейка находит папку с данными. Запустите её ПЕРВОЙ.
import os

CANDIDATES = ["../data", "data", "./data", "/content/data",
              "/content/drive/MyDrive/mglu/data"]
DATA = next((p for p in CANDIDATES if os.path.isdir(p)), None)
print("Data folder:", os.path.abspath(DATA) if DATA else "NOT FOUND")

---

## 1. Индексы и срезы

```
 p  y  t  h  o  n
 0  1  2  3  4  5
-6 -5 -4 -3 -2 -1
```

In [ ]:
word = "python"

print(word[0], word[-1])
print(len(word))

try:
    print(word[6])
except IndexError as error:
    print("IndexError:", error)

In [ ]:
print(word[0:3])     # 'pyt'
print(word[:3])      # то же, начало можно опустить
print(word[3:])      # 'hon'
print(word[-2:])     # 'on'
print(word[::2])     # каждый второй
print(word[::-1])    # в обратном порядке

Длина среза равна `stop - start`.

Срез не выходит за границы и не даёт исключения.

In [ ]:
print(repr("abcd"[0:100]))
print(repr("abcd"[10:20]))
print(repr("abcd"[3:1]))

Удобно, и в этом же ловушка. Опечатка в границах никак себя не проявит,
срез тихо вернёт не тот кусок.

---

## 2. Неизменяемость

In [ ]:
word = "python"

try:
    word[0] = "P"
except TypeError as error:
    print("TypeError:", error)

word = "P" + word[1:]
print(word)

Каждое `+` создаёт новую строку целиком, поэтому склейка в цикле квадратична.

In [ ]:
import time

words = ["word"] * 50_000

start = time.perf_counter()
result = ""
for w in words:
    result = result + w
concat_time = time.perf_counter() - start

start = time.perf_counter()
joined = "".join(words)
join_time = time.perf_counter() - start

print(f"concat in loop: {concat_time:.4f} s")
print(f"join:           {join_time:.6f} s")
print(f"ratio: {concat_time / join_time:.0f}x")
print("results equal:", result == joined)

Увеличьте 50 000 до 100 000. Время `join` вырастет вдвое, время склейки вчетверо.

Правило: копите в списке, склеивайте один раз через `join`.

---

## 3. Методы и проверки

Методы возвращают новую строку и не меняют исходную.

In [ ]:
text = "  Some Mixed Case  "

print(repr(text.strip()))
print(repr(text.strip().lower()))
print(repr(text))          # исходная строка на месте

In [ ]:
text = "  value  "
text.strip()               # результат выброшен
print(repr(text))

text = text.strip()        # так
print(repr(text))

In [ ]:
s = "the quick brown fox"

print(s.replace("quick", "slow"))
print(s.find("brown"))
print(s.find("cat"))            # -1
print(s.count("o"))
print(s.startswith("the"), s.endswith("fox"))

Проверки возвращают `bool`:

In [ ]:
samples = ["12", "ab", "a1", "  ", "a-b", "ёж", "Title"]

for s in samples:
    print(f"{s!r:>8}  isdigit={str(s.isdigit()):<6} "
          f"isalpha={str(s.isalpha()):<6} isspace={s.isspace()}")

`"ёж".isalpha()` истинно, Юникод учитывается. `"a-b".isalpha()` ложно
из-за дефиса. Второе следует из определения буквы, а не из ограничений Python.

---

## 4. `split` и `join`

In [ ]:
print("a b c".split())
print("a,b,c".split(","))
print(" ".join(["a", "b", "c"]))
print("".join(["a", "b", "c"]))

Вызов без аргумента и вызов с пробелом ведут себя по-разному.

In [ ]:
s = "a  b   c"

print(s.split())         # подряд идущие пробелы схлопываются
print(s.split(" "))      # между двумя пробелами пустая строка

Первый вариант нужен почти всегда. Второй даёт пустые элементы,
которые потом всплывают в подсчётах.

In [ ]:
print("key=value=extra".split("=", 1))     # ограничение числа разбиений
print("a\nb\nc".splitlines())

---

## 5. Разбор строки на поля

Прикладной случай, который встречается чаще всего.

In [ ]:
with open(f"{DATA}/dialog.txt", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f if line.strip()]

print(f"строк: {len(lines)}")
for line in lines[:4]:
    print(repr(line))

In [ ]:
def split_speaker(line, separator=":"):
    """Разбивает строку на говорящего и реплику. None, если разделителя нет."""
    if separator not in line:
        return None
    speaker, _, text = line.partition(separator)
    return speaker.strip(), text.strip()


speakers = []
for line in lines:
    parsed = split_speaker(line)
    if parsed is None:
        continue
    speaker, text = parsed
    speakers.append(speaker)

print("реплик разобрано:", len(speakers))
print("говорящие:", sorted(set(speakers)))

`partition` делит строку по первому вхождению и всегда возвращает три части.
Для случая «разделитель может встретиться внутри реплики» это надёжнее `split`.

In [ ]:
# сколько реплик у каждого и какая самая длинная
from collections import Counter

counts = Counter(speakers)
for speaker, n in counts.most_common():
    print(f"{speaker:>10}: {n}")

longest = max(lines, key=len)
print(f"\nсамая длинная строка ({len(longest)} символов):")
print(" ", longest[:100])

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Что напечатает каждая строка?

In [ ]:
# Мой ответ: ...

s = "programming"

# print(s[0:6])
# print(len(s[3:8]))
# print(s[-4:])
# print(s[::-1][:5])
# print("  a b  ".split(" "))
# print(s.find("z"))

### Задача 2. Минимум LeetCode

**LeetCode 709 To Lower Case** плюс письменный разбор.

Задача решается в одну строку, поэтому в разборе ответьте на вопрос поинтереснее:
что `lower()` делает с буквами ниже.

In [ ]:
print("ЁЖ".lower())
print("STRASSE".lower(), "ß".upper())
print("İ".lower(), len("İ".lower()))

### Задача 3. Инициалы

Напишите `initials(full_name)`, превращающую `"Pushkin Alexander Sergeevich"`
в `"Pushkin A. S."`. Обработайте случай, когда частей меньше трёх.

In [ ]:
def initials(full_name):
    # ваш код здесь
    pass

# print(initials("Pushkin Alexander Sergeevich"))
# print(initials("Pushkin Alexander"))
# print(initials("Pushkin"))

### Задача 4. Нормализатор пробелов

Напишите `normalize(text)`, которая убирает пробелы по краям и схлопывает
любые последовательности пробельных символов в один пробел.
Сделайте это без регулярных выражений, только средствами занятия.

In [ ]:
def normalize(text):
    # ваш код здесь
    pass

# print(repr(normalize("  a   b \t\n c  ")))

### Задача 5. Подумать (кода не нужно)

Вы разбираете строки вида `Имя: реплика`. В одной реплике встретилось двоеточие.
Что сделает `split(":")`, что сделает `partition(":")`, что сделает `split(":", 1)`?
Какой вариант правильный и почему.

### Задача 6. Трек «алгоритмы» (по желанию)

344 Reverse String, 557 Reverse Words in a String III, 58 Length of Last Word,
14 Longest Common Prefix.

---

# Итоги

- Индексация с нуля, правая граница среза не включается, длина среза `stop - start`.
- Срез за границами не падает и тихо возвращает не то.
- Строка неизменяема. Методы возвращают новую строку, результат надо присвоить.
- Склейка в цикле квадратична. Копите в списке, склеивайте `join`.
- `split()` и `split(" ")` различаются на подряд идущих пробелах.
- `find` возвращает −1, `index` бросает исключение.
- `partition` надёжнее `split`, когда разделитель может встретиться внутри данных.